# Lab 1 · RAG on a real company handbook

GitLab runs the whole company from a **public handbook** ([handbook.gitlab.com](https://handbook.gitlab.com), CC BY-SA 4.0): thousands of pages of HR, finance, security and legal policy. We take **38 of those pages** as a stand-in for *your* organisation's documents and build an assistant that answers **from the documents, with sources**, says *"I don't know"* when they don't cover the question — and we **measure** which chunking strategy and which metadata filters actually help.

```
index once:   pages → clean → chunk → embed → ChromaDB (+ metadata)
per question: question → [filter on metadata] → top-k chunks → similarity gate → prompt → cited answer
```

### Setting up the LLM

In [1]:
!pip install -q -U transformers accelerate chromadb pypdf sentence-transformers langchain-text-splitters
import torch, json, re
from transformers import AutoTokenizer, AutoModelForImageTextToText

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.

In [2]:
MODEL_ID = "Qwen/Qwen3.5-4B"          # 4B open model, ~9 GB in fp16, fits a free T4. (Fallback: "Qwen/Qwen3-4B")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [3]:
def chat(messages, max_new_tokens=400, temperature=0.0):
    """The one function that talks to the model: a prompt string or a list of {role, content} -> reply text."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)  # wrap in the model's own chat markup
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # drop the prompt, keep only what was generated

In [4]:
print(chat("What is it like to be an LLM? in one sentance"))

Being an LLM is like having a vast library of human knowledge instantly accessible, yet lacking true consciousness, personal experiences, or the ability to feel emotions.


## 0 · The problem: the model has never read your documents

Even when it *has* seen a public page during training, you cannot tell a memory from a guess — and it has certainly never seen your internal ones.

In [5]:
print(chat("At GitLab, what is the daily limit for meals while travelling for work?"))          # a guess, a shrug, or an unverifiable "fact"|

GitLab does not publicly disclose a specific, fixed daily monetary limit for meals while traveling for work in its official travel policy documents.

However, based on standard corporate travel policies and GitLab's general approach to employee benefits, the reimbursement is typically structured as follows:

*   **Per Diem Rates**: Most companies, including GitLab, reimburse travel expenses using a **per diem rate** (a daily allowance) rather than requiring employees to submit receipts for every meal. This rate usually covers both breakfast, lunch, and dinner, plus incidental expenses.
*   **Location-Based Rates**: The specific dollar amount varies significantly depending on the **location** of the travel. Rates are generally higher in major metropolitan areas (e.g., New York, San Francisco, London) and lower in smaller cities or rural areas.
*   **Policy Source**: The exact rates are usually defined by the **US General Services Administration (GSA)** for domestic travel or the **Forei

## 1 · Load the documents — with metadata

Real RAG starts with pulling documents out of a source system (a wiki, SharePoint, a ticketing tool). Here the source is GitLab's handbook repository. For each page we keep **metadata** next to the text: `department`, `country`, `updated` date and `url`. Text answers *what does it say*; metadata answers *which documents may even be considered* — you will see why in section 6.

In [6]:
import requests, re, pandas as pd, concurrent.futures as cf
from urllib.parse import quote

API = "https://gitlab.com/api/v4/projects/gitlab-com%2Fcontent-sites%2Fhandbook/repository"
REF = "df66e66b937d38c1ed4e3dd452927ddf01be58b0"      # one fixed commit of the handbook, so everyone sees the same text
PAGES = """
people-group/time-off-and-absence/time-off-types.md
people-group/time-off-and-absence/leave-types.md
people-group/anti-harassment.md
people-group/contracts-probation-periods.md
people-group/acceptable-use-policy.md
people-group/general-onboarding/_index.md
people-group/offboarding/faq.md
people-policies/inc-usa.md
people-policies/leave-of-absence/us.md
people-policies/ireland-ltd/right-to-disconnect-policy.md
people-policies/france-sas/remote-work-charter.md
total-rewards/benefits/_index.md
total-rewards/benefits/parental-leave-toolkit.md
total-rewards/benefits/general-and-entity-benefits/ltd-benefits-uk.md
total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd.md
total-rewards/benefits/general-and-entity-benefits/bv-benefits-netherlands.md
total-rewards/benefits/general-and-entity-benefits/inc-benefits-us/_index.md
total-rewards/benefits/general-and-entity-benefits/pty-benefits-australia.md
total-rewards/benefits/general-and-entity-benefits/canada-corp-benefits.md
total-rewards/compensation/_index.md
total-rewards/stock-options.md
finance/expenses.md
finance/spending-company-money.md
finance/procurement/individual-use-software.md
finance/accounts-payable/corp-credit-cards.md
security/policies_and_standards/password-standard.md
security/policies_and_standards/data-classification-standard.md
security/policies_and_standards/physical-security-standard-for-company-assets.md
security/policies_and_standards/records-retention-deletion.md
security/policies_and_standards/token-management-standard.md
legal/anti-corruption-policy.md
legal/anti-fraud-policy.md
legal/anti-retaliation-policy.md
legal/policies/gifts-contributions.md
legal/acceptable-use-policy.md
business-technology/_index.md
communication/confidentiality-levels.md
hiring/referral-process.md
""".split()

# Metadata we can derive from the path itself: the folder gives the department, a keyword gives the country.
DEPARTMENT = {"people-group": "People", "people-policies": "People", "total-rewards": "Total Rewards", "finance": "Finance", "security": "Security", "legal": "Legal", "business-technology": "IT", "communication": "Communication", "hiring": "Hiring"}
COUNTRY = {"benefits-uk": "UK", "ireland": "IE", "netherlands": "NL", "benefits-us": "US", "inc-usa": "US", "/us.md": "US", "india": "IN", "france": "FR", "australia": "AU", "canada": "CA"}

def fetch(page):
    """One handbook page -> {title, department, country, updated, url, text}. The metadata is what filters will use later."""
    path = "content/handbook/" + page
    text = requests.get(f"{API}/files/{quote(path, safe='')}/raw", params={"ref": REF}).text      # raw markdown at that commit
    updated = requests.get(f"{API}/commits", params={"path": path, "ref_name": REF, "per_page": 1}).json()[0]["committed_date"][:10]   # last edit date = freshness
    front, body = text.split("\n---\n", 1)                                      # YAML front matter | markdown body
    return {"title": re.search(r'^title:\s*"?(.*?)"?\s*$', front, re.M).group(1),   # title lives in the front matter
            "department": DEPARTMENT[page.split("/")[0]],
            "country": next((c for key, c in COUNTRY.items() if key in page), "global"),   # no keyword matched -> applies everywhere
            "updated": int(updated.replace("-", "")),                          # 20260615 -> numbers can be compared in filters
            "url": "https://handbook.gitlab.com/handbook/" + page.replace("/_index.md", "/").replace(".md", "/"),   # real link, for citations
            "text": body}

with cf.ThreadPoolExecutor(8) as pool:                                          # 8 pages at a time, ~15 seconds
    docs = list(pool.map(fetch, PAGES))
print(len(docs), "pages,", sum(len(d["text"]) for d in docs), "characters")
pd.DataFrame(docs).drop(columns="text").sample(8, random_state=1)               # look at the metadata, not the text

38 pages, 714794 characters


,title,department,country,updated,url
2,Anti-Harassment Policy,People,global,20260206,https://handbook.gitlab.com/handbook/people-gr...
29,GitLab Token Management Standard,Security,global,20260212,https://handbook.gitlab.com/handbook/security/...
3,"Contracts, Background Screenings, Probation Pe...",People,global,20260619,https://handbook.gitlab.com/handbook/people-gr...
22,Spending Company Money,Finance,global,20250130,https://handbook.gitlab.com/handbook/finance/s...
25,GitLab Password Standards,Security,global,20260728,https://handbook.gitlab.com/handbook/security/...
27,Physical Security Standard for Company Assets,Security,global,20260212,https://handbook.gitlab.com/handbook/security/...
21,Global Travel and Expense Policy,Finance,global,20260831,https://handbook.gitlab.com/handbook/finance/e...
35,IT Enterprise Applications,IT,global,20250117,https://handbook.gitlab.com/handbook/business-...


## 2 · Clean

Raw pages carry markup that has nothing to do with meaning (template shortcodes, HTML tags, link URLs). Embedding noise wastes the small context we can give the model, so strip it.

In [7]:
def clean(text):
    text = re.sub(r"\{\{[<%].*?[>%]\}\}|<!--.*?-->|<[^>\n]+>", "", text, flags=re.S)    # template shortcodes, HTML comments and tags
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)                                 # [link text](url) -> link text
    return re.sub(r"[ \t]+", " ", text).strip() # collapse runs of spaces

for d in docs:
    d["text"] = clean(d["text"])
print(docs[0]["title"], "\n", docs[0]["text"][:1000])

Time Off Types 
 For information specific to your location, please review this page.

## Flexible Paid Time Off (PTO)

*This page focuses specifically on GitLab's Flexible PTO policy and is written with a focus on using PTO to fulfill personal obligations, rest or recharge, enjoy a vacation, or spend time with family and friends. You can learn more about GitLab's overall Philosophy to Time Away from Work here and find more detailed policies on all Time Off types and Leave types in their respective handbook sections.*

### Overview

GitLab offers a Flexible PTO policy that empowers team members to take the time they need while ensuring business continuity. This policy reflects our values of transparency, collaboration, and GitLab's overall Philosophy to Time Away from Work.

- Our Flexible PTO policy means that as long as you have coordinated with your manager about your planned time off and made a coverage plan, you are free to take PTO.
- GitLab does not set a specific limit on the am

## 3 · Chunking — four strategies

A page is too long to embed as one vector (one vector cannot mean 30 different things), so we split it. *How* we split is a real design decision:

| Strategy | Idea | Risk |
|---|---|---|
| **tiny** — fixed 100 chars | very precise matching | the answer gets cut in half |
| **medium** — fixed 300 chars | a few sentences per chunk | may split a table or list |
| **huge** — fixed 3,000 chars | lots of context per hit | one vector for many topics: similarity gets diluted |
| **sections** — split on headings, cap at 800 | follows the author's structure, heading travels with the text | sections vary wildly in length |

In [8]:
def chunk_fixed(text, size=300):
    """Split into pieces of about `size` characters, on word boundaries."""
    pieces, current = [], ""
    for word in text.split():
        if len(current) + len(word) > size:        # `size` is a NUMBER of characters, not a chunker function
            pieces.append(current.strip()); current = ""     # close the chunk, start a new one
        current += word + " "
    return pieces + [current.strip()]              # don't lose the last, partial chunk

def chunk_sections(text, max_size=800):
    """Split on markdown headings; keep the heading with its text; cut very long sections with chunk_fixed."""
    pieces = []
    for section in re.split(r"\n(?=#{1,4} )", text):         # lookahead: split before a heading, keep the heading
        heading = section.split("\n")[0] if section.startswith("#") else ""
        pieces += [p if p.startswith(heading) else heading + "\n" + p for p in chunk_fixed(section, max_size)]   # heading travels with every piece
    return [p for p in pieces if len(p) > 60]                # drop scraps too short to mean anything

In [9]:
# Each value is a one-argument chunker, so the indexing loop below can treat them all the same.
STRATEGIES = {"tiny": lambda t: chunk_fixed(t, 100), "medium": lambda t: chunk_fixed(t, 300),
              "huge": lambda t: chunk_fixed(t, 3000), "sections": chunk_sections}


In [10]:
page = next(d for d in docs if d["title"] == "GitLab Password Standards")       # one page, four ways to cut it
for name, chunker in STRATEGIES.items():
    pieces = chunker(page["text"])
    print(f"{name:<9} {len(pieces):>3} chunks | first chunk: {pieces[0][:100]!r}")

tiny       86 chunks | first chunk: '## Purpose This document outlines information security password standards intended to protect GitLab'
medium     29 chunks | first chunk: '## Purpose This document outlines information security password standards intended to protect GitLab'
huge        3 chunks | first chunk: '## Purpose This document outlines information security password standards intended to protect GitLab'
sections   15 chunks | first chunk: '## Purpose This document outlines information security password standards intended to protect GitLab'


## 4 · Embed and index — one collection per strategy

Each chunk becomes a vector with a small embedding model (`all-MiniLM-L6-v2`, 22M parameters, runs on the GPU in seconds) and is stored in **ChromaDB** together with its metadata. Same meaning → nearby vectors, even with different words.

In [11]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedder = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda")   # text -> 384-dim vector, on the GPU
client = chromadb.Client()                                                                      # in-memory DB: gone when the runtime restarts

def index(name, chunker):
    kb = client.create_collection(name, embedding_function=embedder, metadata={"hnsw:space": "cosine"})   # cosine: compare direction, not length
    for n, d in enumerate(docs):
        pieces = chunker(d["text"])
        kb.add(ids=[f"{n}-{i}" for i in range(len(pieces))], documents=pieces,        # ids must be unique within the collection
               metadatas=[{k: d[k] for k in ("title", "department", "country", "updated", "url")}] * len(pieces))   # every chunk inherits its page's metadata
    return kb

KBS = {name: index(name, chunker) for name, chunker in STRATEGIES.items()}     # four collections, same documents, different cuts
{name: kb.count() for name, kb in KBS.items()}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'tiny': 6412, 'medium': 2115, 'huge': 226, 'sections': 1160}

## 5 · Search by meaning — and measure which chunking works

`search()` embeds the question and returns the k nearest chunks with a similarity score (1 = identical meaning). First look at one question across the four strategies, then **measure**: fourteen questions whose answers we know; does the answer text appear in the top-3 chunks?

In [12]:
def search(kb, question, k=3, where=None):
    """[(similarity, metadata, chunk_text), ...] for the k most similar chunks; `where` filters on metadata first."""
    r = kb.query(query_texts=[question], n_results=k, where=where)    # the question is embedded with the same model as the chunks
    return [(round(1 - dist, 2), meta, doc) for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

In [13]:
q = "What is the daily limit for meals while travelling for work?"
for name, kb in KBS.items():
    score, meta, text = search(kb, q, 1)[0]
    print(f"{name:<9} {score}  {meta['title'][:]:<30} | {text[:500]!r}")

tiny      0.66  Global Travel and Expense Policy | 'lodging and meals during the part of the trip that is work-related. Depending on the distance of'
medium    0.56  Global Travel and Expense Policy | 'representative (including any government prospects and government customers). - Meals are otherwise reimbursable, however Team Members should be mindful of budget while claiming meals with clients and partners. - $80 limit per person per meal. - All Participants should be added (both internal and'
huge      0.35  GitLab France S.A.S. Remote Work Charter | "of the management. For remote work days, the working time is counted according to a self-reporting system, the monthly balance of which must be submitted to the manager without request at the end of each month. The Payroll handbook page provides more information in this regard with the Global Upside - France section. 5.2 *Modalities for workload regulation* Remote work must not increase the team member's usual workload or jeopardise th

In [14]:
EVAL = [  # (question, text that must appear in the retrieved context)
    ("How many days of time off should I take at minimum each year?", "25 days"),
    ("What percentage of my salary does the company pay into the UK pension?", "4%"),
    ("Which private medical insurance provider covers team members in Ireland?", "VHI"),
    ("What is the daily limit for meals while travelling for work?", "$120"),
    ("How many weeks of paid parental leave do team members get?", "16 weeks"),
    ("What is the minimum password length?", "12 characters"),
    ("How long can a personal access token be valid before it expires?", "30 days"),
    ("Can I install Chrome extensions on my work laptop?", "Chrome Extensions"),
    ("What is the maximum value of a gift to a US government official per event?", "$20"),
    ("How do I return my laptop when I leave?", "prepaid shipping label"),
    ("How long are production change records retained?", "3 years"),
    ("Are interns eligible for referral bonuses?", "not eligible"),
    ("How many vacation days are Netherlands team members entitled to?", "20 vacation days"),
    ("What is the reimbursement limit for internet while travelling?", "$50"),
]

In [15]:
def evaluate(kb, k=3):
    # This measures RETRIEVAL only — the model is not involved yet. If the answer text isn't here, no prompt can rescue it.
    found = sum(any(ans.lower() in text.lower() for _, _, text in search(kb, q, k)) for q, ans in EVAL)
    avg = sum(len(c) for c in kb.get()["documents"]) // kb.count()       # average chunk size
    return {"chunks": kb.count(), "avg chars": avg, "context per question": avg * k,   # what each strategy costs in prompt space
            "answer found in top-3": f"{found}/{len(EVAL)}", "mean top similarity": round(sum(search(kb, q, 1)[0][0] for q, _ in EVAL) / len(EVAL), 2)}

pd.DataFrame({name: evaluate(kb) for name, kb in KBS.items()}).T

,chunks,avg chars,context per question,answer found in top-3,mean top similarity
tiny,6412,96,288,7/14,0.67
medium,2115,293,879,12/14,0.63
huge,226,2757,8271,10/14,0.49
sections,1160,546,1638,11/14,0.62


Read the table across: **tiny** chunks score the *highest similarity* yet find the *fewest answers* — the matching phrase is there but the number next to it fell into the neighbouring chunk. **huge** chunks carry about 8,000 characters of context per question, slow the model down, and still miss answers because one vector now stands for many topics. **medium** and **sections** find the most with a fraction of the context. None of this is a law of nature: **the right chunking depends on your documents and your questions — measure it like this, do not copy it.**

We continue with `medium`.

In [16]:
kb = KBS["medium"]

## 6 · Metadata filters — change *which documents compete*

Similarity search ranks chunks; a `where` filter first decides **which chunks are allowed to compete**. Metadata is exact and cheap, so use it whenever the question implies it: the user's country, a date range, a department.

In [17]:
q = "Which private medical insurance provider do I get?"
print("no filter   :", [(s, m["title"][:28]) for s, m, _ in search(kb, q)])              # every country's page competes
print("country=UK  :", [(s, m["title"][:28]) for s, m, _ in search(kb, q, where={"country": "UK"})])
print("country=IE  :", [(s, m["title"][:28]) for s, m, _ in search(kb, q, where={"country": "IE"})])

no filter   : [(0.51, 'GitLab Inc (US) Benefits'), (0.5, 'GitLab Inc (US) Benefits'), (0.48, 'PTY LTD Benefits')]
country=UK  : [(0.45, 'GitLab LTD (UK) Benefits'), (0.42, 'GitLab LTD (UK) Benefits'), (0.4, 'GitLab LTD (UK) Benefits')]
country=IE  : [(0.44, 'GitLab Ireland Ltd'), (0.42, 'GitLab Ireland Ltd'), (0.41, 'GitLab Ireland Ltd')]


In [18]:
q = "How quickly must I submit an expense after buying something?"
print("all pages              :", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q)])
print("updated since June 2026:", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q, where={"updated": {"$gte": 20260601}})])
print("Finance only           :", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q, where={"department": "Finance"})])
print("department=Legal       :", [(s, m["title"][:26]) for s, m, _ in search(kb, "Can I install Chrome extensions on my laptop?", where={"department": "Legal"})])

all pages              : [(0.69, 'Spending Company Money', 20250130), (0.52, 'Global Travel and Expense ', 20260831), (0.49, 'Global Travel and Expense ', 20260831)]
updated since June 2026: [(0.52, 'Global Travel and Expense ', 20260831), (0.49, 'Global Travel and Expense ', 20260831), (0.48, 'Global Travel and Expense ', 20260831)]
Finance only           : [(0.69, 'Spending Company Money', 20250130), (0.52, 'Global Travel and Expense ', 20260831), (0.49, 'Global Travel and Expense ', 20260831)]
department=Legal       : [(0.11, 'Anti-Corruption Policy'), (0.11, 'Policies related to Gifts '), (0.1, 'Policies related to Gifts ')]


Three effects worth remembering: the **country filter turns an ambiguous question into a precise one** (without it, the US and Australian pages outrank the UK's); a **"recent pages only" filter silently dropped the best page** ("Spending Company Money" was last edited in January 2025) — filters hide as well as focus, so derive them from what the user actually needs; and the **wrong department filter leaves only irrelevant chunks** with tiny similarity — which is exactly what the gate in the next section catches.

## 7 · Generate a grounded answer — the whole pipeline in one function

Retrieve (with optional filter) → check the best similarity (refuse junk) → paste the chunks into the prompt as numbered passages → ask for citations → append the real sources.

In [19]:
def answer(question, k=3, where=None, threshold=0.35):
    hits = search(kb, question, k, where)
    if not hits or hits[0][0] < threshold:                                   # nothing relevant -> do not let the model improvise
        return f"I don't have that information in the handbook. (best similarity {hits[0][0] if hits else 0})"
    context = "\n\n".join(f"[{i}] ({m['title']}) {text}" for i, (_, m, text) in enumerate(hits, 1))
    reply = chat([{"role": "system", "content": "Answer ONLY from the context passages. Cite them like [1]. If the context does not contain the answer, say so."},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    sources = "\n".join(f"[{i}] {m['title']} — {m['url']}" for i, (_, m, _) in enumerate(hits, 1))
    return f"{reply}\n\nSources:\n{sources}"

print(answer("What is the daily limit for meals while travelling for work?"))

The daily limit for meals while travelling for work is $80 per person per meal [1][2].

Sources:
[1] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[2] Global Travel and Expense Policy — https://handbook.gitlab.com/handbook/finance/expenses/
[3] GitLab France S.A.S. Remote Work Charter — https://handbook.gitlab.com/handbook/people-policies/france-sas/remote-work-charter/


In [20]:
print(answer("Which private medical insurance provider do I get, and who pays the premium?", where={"country": "IE"}), "\n")
print(answer("Who won the 2022 World Cup?"), "\n")                                                    # out of scope -> refused, not invented
print(answer("Can I install Chrome extensions on my laptop?", where={"department": "Legal"}))          # wrong filter -> refused, not invented

Based on the context provided, the private medical insurance provider is **VHI** [2], and **GitLab Ireland Ltd** pays 100% of the premiums for you and your eligible dependents [1].

Sources:
[1] GitLab Ireland Ltd — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd/
[2] GitLab Ireland Ltd — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd/
[3] GitLab Ireland Ltd — https://handbook.gitlab.com/handbook/total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd/ 

I don't have that information in the handbook. (best similarity 0.19) 

I don't have that information in the handbook. (best similarity 0.11)


Compare with step 0: same model, now correct, sourced and refusing what it cannot back up. **Where does 0.35 threshold in our function (answer) come from?** From measuring, not guessing: in-scope questions land clearly above out-of-scope ones, and the threshold goes in the gap. It changes with the embedding model, the chunking and your documents — re-measure whenever any of them changes. (Watch *"the capital of France"*: it scores close to the gate because a French policy exists. Thresholds catch junk, not every wrong match.)

In [21]:
OUT_OF_SCOPE = ["Who won the 2022 World Cup?", "How do I fix a flat bicycle tyre?", "Best pizza recipe?", "What is the capital of France?"]
print("in scope  :", sorted(search(kb, q, 1)[0][0] for q, _ in EVAL))
print("out scope :", sorted(search(kb, q, 1)[0][0] for q in OUT_OF_SCOPE))

in scope  : [0.43, 0.52, 0.55, 0.56, 0.57, 0.58, 0.63, 0.64, 0.67, 0.69, 0.71, 0.75, 0.77, 0.79]
out scope : [0.17, 0.19, 0.25, 0.35]


## 8 · 🗂️ Your turn: your own documents

Upload 1–5 files (`.pdf .txt .md .csv`) from your organisation — a policy, a report, a manual. They stay inside this Colab session. The same clean → chunk → embed → search → answer pipeline runs on them; you become the source of truth for whether the answers are right.

In [22]:
from google.colab import files
from pypdf import PdfReader

def upload():
    """Open the file picker; return {filename: text} for .pdf / .txt / .md / .csv files."""
    texts = {}
    for name, blob in files.upload().items():
        if name.lower().endswith(".pdf"):
            open(name, "wb").write(blob)
            texts[name] = "\n".join(page.extract_text() or "" for page in PdfReader(name).pages)
        else:
            texts[name] = blob.decode("utf-8", errors="ignore")
    print({k: f"{len(v)} chars" for k, v in texts.items()})
    return texts

In [23]:
mine = upload()

client.delete_collection("my_documents") if "my_documents" in [c.name for c in client.list_collections()] else None
my_kb = client.create_collection("my_documents", embedding_function=embedder, metadata={"hnsw:space": "cosine"})
for name, text in mine.items():
    pieces = chunk_sections(clean(text))
    my_kb.add(ids=[f"{name}-{i}" for i in range(len(pieces))], documents=pieces,
              metadatas=[{"title": name, "department": "mine", "country": "global", "updated": 20260907, "url": name}] * len(pieces))
print(my_kb.count(), "chunks from your files")

{}
0 chunks from your files


In [24]:
kb = my_kb                                                                       # point answer() at your documents

In [25]:
while True:                                                                      # ask anything, 'quit' to stop
    q = input("Ask your documents: ").strip()
    if q.lower() in ("quit", "exit", "q", ""): break
    print(answer(q), "\n")

Ask your documents: q


If a question you *know* is answered comes back as "I don't have that information", the best similarity was below the threshold: your wording differs from the document's. Rephrase, or lower `threshold=0.25`. If the answer is wrong, print `search(kb, q)` — nine times out of ten the retrieval, not the model, is the problem.

## Exercises · Recap
1. **Overlap**: write `chunk_overlap(text, size=300, overlap=60)` (each chunk repeats the last 80 characters of the previous one), add it to `STRATEGIES`, re-run the table. Does it rescue the answers `tiny` lost?
2. **k**: re-run `evaluate(kb, k=1)` and `k=6`. When does more context help, when is it noise?
3. **Follow-ups**: ask *"And in the Netherlands?"* after the pension question — it fails. Fix it with one `chat()` call that rewrites the follow-up into a standalone question first.
4. **Keyword + vector**: ChromaDB also has `where_document={"$contains": "pension"}`. Combine it with the vector search for questions that need an exact word (a product name, an error code).

| Component | In this lab | Production upgrade |
|---|---|---|
| Loader + metadata | `fetch()` from the handbook API | connectors (SharePoint, Confluence, tickets), access control in metadata |
| Cleaner | `clean()` — 3 regexes | format-aware parsers (PDF layout, tables) |
| Chunker | `chunk_fixed` / `chunk_sections` + **measured** | parent–child chunks, overlap, per-document-type rules |
| Embeddings + vector DB | `all-MiniLM-L6-v2` + ChromaDB | bigger embedder, hybrid keyword + vector search, reranker |
| Retriever | `search()` with `where` filters | filters from the user's identity and question (country, date, permissions) |
| Generator + gate | `answer()` with a measured threshold | evaluation set: hit rate, faithfulness, answer quality |

*Handbook text © GitLab Inc., [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).* **Next → Lab 2:** the *agent* decides which tool to call.